# MDTF Variable Translation: Developer Guide

> Disclaimer: This notebook was created with GitHub Copilot assistance and is still being vetted.

This notebook introduces the framework's variable-name translation machinery. It is for code exploration and small examples; it does not launch a full MDTF run or modify the fieldlist files. Run it from a kernel with the MDTF repository dependencies available.

## The Main Idea

POD settings describe requested variables. Fieldlist tables describe names and metadata used by data conventions. The translator loads those tables and routes lookups to the fieldlist for a convention. In a framework run, `translate_data` decides whether the data convention is used or whether the special `no_translation` fieldlist is selected.

The main runtime path is:

1. [`mdtf_framework.py`](../../mdtf_framework.py#L159) loads translator tables and [`mdtf_framework.py`](../../mdtf_framework.py#L185) starts POD setup.
2. [`PodObject.setup_pod`](../../src/pod_setup.py#L313) checks `runtime_config.translate_data` and chooses the target convention.
3. [`DataSourceBase.translate_varlist`](../../src/data_sources.py#L100) forwards the variable to its varlist.
4. [`Varlist.setup_var`](../../src/varlist_util.py#L547) selects a fieldlist and calls that fieldlist's `translate()` method.

The `VariableTranslator.translate()` wrapper is a separate convenience method. The gated framework path above calls the selected fieldlist directly. See [`translation.py`](../../src/translation.py#L634) for the wrapper implementation.

## Find The Repository Root

The source package is named `src`. Find the checkout from the notebook's current directory so imports work whether Jupyter started in the repository root or a subdirectory.

In [4]:
import sys
from pathlib import Path

candidate_dirs = (Path.cwd(), *Path.cwd().parents)
repo_root = next(
    (path for path in candidate_dirs if (path / "src" / "translation.py").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("Start Jupyter in the MDTF checkout or one of its subdirectories.")

sys.path.insert(0, str(repo_root))
print(repo_root)

/work5/j3k/mdtf_main/MDTF-diagnostics


## Load The Fieldlists

`VariableTranslator` is a singleton in this codebase. The framework initializes it once with `CODE_ROOT`, then reads `data/fieldlist_*.jsonc`. For an isolated notebook kernel, use the same initialization sequence below.

In [5]:
from src.translation import VariableTranslator

translator = VariableTranslator(str(repo_root))
translator.read_conventions(str(repo_root))

print("Loaded conventions:", sorted(translator.conventions.keys()))

Loaded conventions: ['cesm', 'cmip', 'gfdl']


## Verify The GFDL-to-CMIP Name Mapping

First ask the translator for the CF standard name of GFDL `t_ref`. Then look up that standard name in CMIP using the same realm and modifier. This isolates the fieldlist mapping before translating a full `VarlistEntry`.

In [13]:
gfdl_t_ref = translator.get_convention("GFDL").lut["t_ref"]
cmip_tas = translator.get_convention("CMIP").lut["tas"]

print("GFDL t_ref:", gfdl_t_ref["standard_name"], gfdl_t_ref["modifier"])
print("CMIP tas:", cmip_tas["standard_name"], cmip_tas["modifier"])
assert gfdl_t_ref["standard_name"] == cmip_tas["standard_name"]
assert gfdl_t_ref["realm"] == cmip_tas["realm"]
assert gfdl_t_ref["modifier"] == cmip_tas["modifier"]

GFDL t_ref: air_temperature atmos_height
CMIP tas: air_temperature atmos_height


## Translate A GFDL Varlist Entry To CMIP

At runtime, `Varlist.setup_var()` gets the CMIP `Fieldlist` and calls its `translate(var, from_convention)` method. The example below follows that exact route with a GFDL `VarlistEntry` and the CMIP target fieldlist. The matching entries share `air_temperature`, realm `atmos`, units `K`, three dimensions, and modifier `atmos_height`, so the translated name should be `tas`.

For standalone use, `VariableTranslator.translate(var, to_convention, from_convention)` is a wrapper that selects the fieldlist and calls the same method.

In [12]:
from types import SimpleNamespace
from src.varlist_util import Varlist

example_settings = {
    "data": {"frequency": "day", "realm": "atmos"},
    "dimensions": {
        "lat": {
            "axis": "Y",
            "standard_name": "latitude",
            "units": "degrees_north",
        },
        "lon": {
            "axis": "X",
            "standard_name": "longitude",
            "units": "degrees_east",
        },
        "time": {"axis": "T", "standard_name": "time", "units": "days"},
    },
    "varlist": {
        "t_ref": {
            "standard_name": "air_temperature",
            "long_name": "temperature at 2 m",
            "units": "K",
            "modifier": "atmos_height",
            "dimensions": ["time", "lat", "lon"],
        }
    },
}
pod = SimpleNamespace(
    pod_data=example_settings["data"],
    pod_dims=example_settings["dimensions"],
    pod_vars=example_settings["varlist"],
    pod_settings={"convention": "GFDL"},
    _log_name="translation_notebook",
)
example_var = Varlist.from_struct(pod).vars[0]
cmip_fieldlist = translator.get_convention("CMIP")
translated_var = cmip_fieldlist.translate(example_var, "GFDL")
translated_via_wrapper = translator.translate(
    example_var, to_convention="CMIP", from_convention="GFDL"
)

print("Requested:", example_var.name, example_var.standard_name)
print("Framework path:", translated_var.name, translated_var.standard_name)
assert translated_var.name == "tas"
assert translated_via_wrapper.name == translated_var.name

Requested: t_ref air_temperature
Framework path: tas air_temperature


## How The Runtime Gate Works

In [`src/pod_setup.py`](../../src/pod_setup.py#L313), each case starts with its configured data convention. If `translate_data` is false, setup substitutes `no_translation`; otherwise the configured data convention remains the target. Setup passes the POD convention and chosen target convention into `translate_varlist()`.

For the worked translation here, those values are `pod_convention="GFDL"`, `data_convention="CMIP"`, and `translate_data=True`. The framework therefore selects the CMIP fieldlist and passes the GFDL convention as the source.

In [`src/varlist_util.py`](../../src/varlist_util.py#L547), `VariableTranslator().get_convention(to_convention)` returns the data fieldlist, or a `NoTranslationFieldlist` for the sentinel. The code then calls `translate.translate(v, from_convention)`. So the switch affects which fieldlist implementation runs; it does not skip the varlist setup call.

Keep the two APIs distinct when debugging:

- `VariableTranslator.translate(var, to_convention, from_convention)` is a wrapper that chooses a fieldlist and calls its `translate()`.
- The `translate_data` runtime path calls `Fieldlist.translate(var, from_convention)` on the fieldlist already selected in `Varlist.setup_var()`.
- Preprocessor request-editing has a direct `VariableTranslator.translate()` call at [`src/preprocessor.py`](../../src/preprocessor.py#L206); that is another use case, not the setup gate.

## Where To Go Next

- Read [`src/translation.py`](../../src/translation.py#L517) for convention loading, aliases, and fieldlist routing.
- Read [`src/translation.py`](../../src/translation.py#L363) for the convention-specific `Fieldlist.translate()` implementation.
- Inspect [`data/fieldlist_CMIP.jsonc`](../../data/fieldlist_CMIP.jsonc) and the other `fieldlist_*.jsonc` files to understand translation metadata.
- Run the focused tests with `python -m unittest src.tests.test_translation.TestVariableTranslatorFiles.test_variabletranslator_real_data` from the repository root.
- For a broader notebook example using catalog translations, see [`example_multirun_demo_vartrans.ipynb`](example_multirun_demo_vartrans.ipynb).